##### Adding Tables in Postgres SQL
`docker exec -i db psql -U admin -d shop < seed.sql`
##### Running SQL in Docker
`docker exec -it db psql -U admin -d shop -c "SELECT * FROM customers;`

#### Agents For Tool Context, State and Memory

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()
from dataclasses import dataclass
from langchain.agents import create_agent, AgentState
from langchain.tools import tool, ToolRuntime
from langgraph.store.memory import InMemoryStore
from langgraph.checkpoint.memory import InMemorySaver
from langchain_community.utilities import SQLDatabase

@dataclass
class AnanlystContext:
    user_id:str
    role: str

class AnalystState(AgentState):
    last_query: str | None = None
    row_count: int | None = None
    
DB_URL = os.getenv("DB_URL")
db = SQLDatabase.from_uri(DB_URL)

@tool
def run_sql(sql: str, runtime: ToolRuntime[AnalystState, AnanlystContext]) -> str:
     """Execute a SELECT query against the shop database.
     
        use this when the user asks about customers, orders, revenue, etc.
        Only SELECT statements are allowed.
     """
     if runtime.context.role == "viewer":
         forbidden = ["DELETE", "UPDATE", "INSERT", "CREATE", "ALTER", "DROP"]
         if any(word in sql.upper() for word in forbidden):
             return f"Access denied: '{runtime.context.user_id}' cannot modify data."
     try:
        result = db.run(sql)
        return result
     except Exception as e:
         return f"Query error {e}"
@tool     
def list_tables(runtime: ToolRuntime[AnanlystContext, AnalystState]) -> str:
     """List all available tables. Call this BEFORE writing SQL to understand the schema."""
     return db.get_table_info()
@tool
def recap_last_query(runtime: ToolRuntime[AnanlystContext, AnalystState]) -> str:
    """Tell me what I asked last time in this conversation."""
    last = runtime.state.get('last_query')
    count = runtime.state.get('row_count')
    if not last:
        return "No queries run yet in this conversation."
    return f"Last query: {last}\nRows returned: {count}"
     
agent = create_agent(
    model="ollama:qwen2.5:7b",
    tools=[run_sql, list_tables, recap_last_query],
    context_schema=AnanlystContext,
    state_schema=AnalystState,
    checkpointer=InMemorySaver(),
    store=InMemoryStore(),
    system_prompt=(
        "You are a data analyst assistant. "
        "ALWAYS call list_tables first to understand the schema. "
        "Write safe SELECT queries with LIMIT clauses. "
        "Never modify data unless explicitly told."
    ),
)
config = {"configurable": {"thread_id": "session-1"}}
context = AnanlystContext(user_id="alice", role="viewer")
# Question 1: What tables do we have?
q1 = "What tables do we have?"
result = agent.invoke(
    {"messages": [{"role": "user", "content": q1}]},
    config=config,
    context=context,
)
print("❓ Q:", q1)
print("🤖 A:", result["messages"][-1].content)
print()


# Question 2: Top spending customers?
q2 = "Which customers spent the most money?"
result = agent.invoke(
    {"messages": [{"role": "user", "content": q2}]},
    config=config,
    context=context,
)
print("❓ Q:", q2)
print("🤖 A:", result["messages"][-1].content)
print()

# Question 3: Use STATE — what was my last query?
q3 = "What was my last query about?"
result = agent.invoke(
    {"messages": [{"role": "user", "content": q3}]},
    config=config,
    context=context,
)
print("❓ Q:", q3)
print("🤖 A:", result["messages"][-1].content)
print()

# Question 4: Try a write (should be blocked for viewer)
q4 = "Delete all orders from Bob"
result = agent.invoke(
    {"messages": [{"role": "user", "content": q4}]},
    config=config,
    context=context,
)
print("❓ Q:", q4)
print("🤖 A:", result["messages"][-1].content)